# Notebook 2 — Features Deep Dive

A `Feature` is the atomic unit in Blueprint — it describes a single column: what type of data it holds, the shape of its distribution, and any post-generation transforms.

This notebook covers every `dtype` and every modifier method in detail. By the end you'll know exactly which Feature variant to reach for and how to layer modifiers to produce precisely the values you need.

In [1]:
from blueprint import Blueprint, Feature, Influence

import pandas as pd
import numpy as np

## Feature anatomy

Every `Feature` is constructed with the same call signature:

```python
Feature(
    name,          # str  — column name in the output DataFrame
    dtype,         # type or str — what kind of data to generate
    base=0,        # float — center of the distribution (mean for numeric types)
    std=0,         # float — spread of the distribution (σ for numeric types)
    clip=(None, None),  # (lo, hi) — hard bounds applied after all modifiers
    p=0.5,         # float — probability of True for bool features
    values=None,   # list  — category labels (required for dtype='category')
    weights=None,  # list  — sampling weights for values (optional)
    derived=False, # bool  — feature starts at zero; value comes from influences only
    nullable=0.0,  # float — fraction of rows to set to NaN/None
    seed=None,     # int   — per-feature RNG seed (overrides Blueprint seed)
    **kwargs,      # dtype-specific extras: style, start, end, template, pools, formula …
)
```

Most parameters are ignored unless the `dtype` needs them. `base` and `std` only apply to numeric types; `p` only applies to `bool`; `values` only applies to `category`.

In [2]:
f = Feature('salary', dtype=float, base=65_000, std=20_000, clip=(25_000, None))

print('name:    ', f.name)
print('dtype:   ', f.dtype)
print('base:    ', f.base)
print('std:     ', f.std)
print('clip:    ', f._constructor_clip)
print('nullable:', f.nullable)
print('derived:', f.derived)
print('modifiers:', f.modifiers)

name:     salary
dtype:    <class 'float'>
base:     65000
std:      20000
clip:     (25000, None)
nullable: 0.0
derived: False
modifiers: []


## Numeric features: `float` and `int`

Numeric features draw from a normal distribution parameterised by `base` (mean) and `std` (standard deviation). The only difference between `float` and `int` is that `int` rounds the result and casts to `int64`.

**`clip` is applied last**, after all modifiers, so it acts as an absolute floor/ceiling on the final value. Pass `None` to leave one end unbounded.

In [ ]:
bp = Blueprint(n=8, seed=42)
bp.add_feature(Feature('salary', dtype=float, base=65_000, std=20_000, clip=(25_000, None)))

df = bp.emit()
print(df)
print()
print(df['salary'].describe().round(2))

         salary
0  71094.341595
1  44200.317875
2  80009.023916
3  83811.294328
4  25979.296227
5  38956.409863
6  67556.808063
7  58675.148153

count        8.00
mean     58785.33
std      20658.28
min      25979.30
25%      42889.34
50%      63115.98
75%      73323.01
max      83811.29
Name: salary, dtype: float64


In [ ]:
bp = Blueprint(n=8, seed=42)
bp.add_feature(Feature('age', dtype=int, base=35, std=10, clip=(18, 80)))

df = bp.emit()
print(df)
print()
print('dtype:', df['age'].dtype)

   age
0   38
1   25
2   43
3   44
4   18
5   22
6   36
7   32

dtype: int64


### Convenience numeric subtypes

Two additional string dtypes exist for common cases:

| dtype | What it does |
|---|---|
| `"positive_float"` | Normal draw, then hard-clips at 0 (never negative) |
| `"percentage"` | Normal draw, then clips to `[0.0, 1.0]` |

Both accept `base` and `std` the same way as `float`.

In [ ]:
bp = Blueprint(n=200, seed=42)
bp.add_feature(
    Feature('conversion_rate', dtype='percentage',    base=0.12, std=0.05),
    Feature('response_time_s', dtype='positive_float', base=2.5,  std=1.5),
)

df = bp.emit()
print(df.describe().round(4))
print()
print('conversion_rate always in [0,1]:', df['conversion_rate'].between(0, 1).all())
print('response_time_s always >= 0:    ', (df['response_time_s'] >= 0).all())

       conversion_rate  response_time_s
count         200.0000         200.0000
mean            0.1185           2.5603
std             0.0441           1.4691
min             0.0134           0.0000
25%             0.0872           1.4555
50%             0.1174           2.5907
75%             0.1468           3.4587
max             0.2657           6.8576

conversion_rate always in [0,1]: True
response_time_s always >= 0:     True


## Boolean features

`dtype=bool` generates a column of `True`/`False` values. The `p` parameter controls the probability that any given row is `True`.

In [ ]:
bp = Blueprint(n=200, seed=42)
bp.add_feature(
    Feature('is_premium',  dtype=bool, p=0.20),  # 20% True
    Feature('email_opted', dtype=bool, p=0.75),  # 75% True
)

df = bp.emit()
print(df.head(8))
print()
print('is_premium %:  ', f"{df['is_premium'].mean():.1%}")
print('email_opted %: ', f"{df['email_opted'].mean():.1%}")
print('dtypes:', df.dtypes.to_dict())

  is_premium  email_opted
0      False        False
1      False        False
2      False         True
3      False         True
4       True         True
5      False         True
6      False         True
7      False         True

is_premium %:   21.5%
email_opted %:  74.0%
dtypes: {'is_premium': dtype('bool'), 'email_opted': dtype('bool')}


## Categorical features

`dtype="category"` draws from a fixed set of labels. The output column has pandas `CategoricalDtype`, which is memory-efficient for low-cardinality columns.

**Required:** `values` — the list of category labels.  
**Optional:** `weights` — relative sampling probabilities. Without `weights` all labels are equally likely.

In [ ]:
bp = Blueprint(n=200, seed=42)
bp.add_feature(
    Feature('region', dtype='category', values=['North', 'South', 'East', 'West']),
)

df = bp.emit()
print(df['region'].value_counts())
print()
print('dtype:', df['region'].dtype)
print('categories:', list(df['region'].cat.categories))

region
South    53
West     51
East     50
North    46
Name: count, dtype: int64

dtype: category
categories: ['North', 'South', 'East', 'West']


Use `weights` when the distribution across categories should be uneven. The weights don't need to sum to 1 — Blueprint normalises them automatically.

In [ ]:
bp = Blueprint(n=500, seed=42)
bp.add_feature(
    Feature('plan', dtype='category',
            values=['free', 'basic', 'pro', 'enterprise'],
            weights=[60, 25, 12, 3]),
)

df = bp.emit()
counts = df['plan'].value_counts()
print(counts)
print()
print((counts / len(df)).round(3).rename('share'))

plan
free          300
basic         134
pro            56
enterprise     10
Name: count, dtype: int64

plan
free          0.600
basic         0.268
pro           0.112
enterprise    0.020
Name: share, dtype: float64


## Identity features

`dtype="id"` generates unique identifier columns. Three styles are available:

| `style` | Output | Extra params |
|---|---|---|
| `"uuid4"` (default) | Random UUID v4 strings | — |
| `"sequential"` | Integer sequence | `start`, `step` |
| `"prefixed"` | Prefixed string sequence | `prefix`, `start`, `padding` |

In [ ]:
# UUID4 — the default style
bp = Blueprint(n=5, seed=42)
bp.add_feature(Feature('id', dtype='id'))

df = bp.emit()
print(df)

                                     id
0  8826d916-cdfb-41c6-81ff-91a761565a70
1  2416da6e-c212-4ddb-8d88-00160eb686b2
2  eb819333-b501-4c18-8c53-c786ed62c2f9
3  71445abc-2f0d-4ac2-8097-acb7a3823bc9
4  13d16283-160e-4c20-aebd-f9d6297e4c73


In [ ]:
# Sequential integers and zero-padded prefixed strings
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature('row_id',    dtype='id', style='sequential', start=1001),
    Feature('ticket_id', dtype='id', style='prefixed', prefix='TKT-', start=1, padding=5),
)

df = bp.emit()
print(df)

   row_id  ticket_id
0    1001  TKT-00001
1    1002  TKT-00002
2    1003  TKT-00003
3    1004  TKT-00004
4    1005  TKT-00005
5    1006  TKT-00006


Identity features are deterministic within a Blueprint: the same `seed` always produces the same UUIDs and the same sequence.

## Datetime features

`dtype="datetime"` generates timestamps uniformly distributed between `start` and `end`. Both are required and should be date or datetime strings parseable by `pd.Timestamp`.

The output column has `datetime64[ns]` dtype and is compatible with all pandas date operations.

In [ ]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature('signup_date', dtype='datetime', start='2023-01-01', end='2024-12-31'),
)

df = bp.emit()
print(df)
print()
print('dtype:', df['signup_date'].dtype)
print('range:', df['signup_date'].min().date(), 'to', df['signup_date'].max().date())

                    signup_date
0 2024-07-18 23:42:35.894521726
1 2023-11-17 09:09:00.952041451
2 2024-09-18 18:38:08.004650720
3 2024-05-24 01:53:16.328832201
4 2023-03-10 17:59:13.685969833
5 2024-12-13 04:54:12.962433473

dtype: datetime64[ns]
range: 2023-03-10 to 2024-12-13


## Text / template features

`dtype="str"` generates strings from a Python format template. Two extra kwargs are required:

- `template` — a format string with `{placeholder}` slots
- `pools` — a `dict` mapping placeholder names to `Feature` objects

Each pool `Feature` is generated independently for every row. The values produced for the pool features are **not** the same columns that appear in the DataFrame — they're sampled fresh for each string.

This means `{first}` in the email template below draws from the same distribution as the `first` column, but the two columns won't necessarily match row-for-row.

In [ ]:
first_name = Feature('first', dtype='category', values=['Alice', 'Bob', 'Carol', 'Dave', 'Eve'])
dept       = Feature('dept',  dtype='category', values=['Eng', 'Sales', 'Finance', 'HR'])

bp = Blueprint(n=6, seed=42)
bp.add_feature(
    first_name,
    dept,
    Feature('email', dtype='str',
            template='{first}@{dept}.corp',
            pools={'first': first_name, 'dept': dept}),
)

df = bp.emit()
print(df)

   first     dept               email
0  Alice      Eng    Eve@Finance.corp
1   Dave  Finance       Carol@HR.corp
2   Dave      Eng  Carol@Finance.corp
3  Carol      Eng         Bob@HR.corp
4  Carol  Finance  Alice@Finance.corp
5    Eve       HR        Eve@Eng.corp


## Computed features

`dtype="computed"` defers column generation until all other columns exist. A `formula` kwarg receives the current DataFrame and returns the new column's values.

Computed features are always evaluated **last**, so they have access to every other column — including columns modified by classes and influences.

In [ ]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature('price', dtype=float, base=500_000, std=100_000, clip=(150_000, None)),
    Feature('sqft',  dtype=int,   base=2_000,   std=400,     clip=(700, None)),
    Feature('price_per_sqft', dtype='computed',
            formula=lambda df: df['price'] / df['sqft']),
)

df = bp.emit()
print(df.round(2))

       price  sqft  price_per_sqft
0  530471.71  2051          258.64
1  396001.59  1874          211.31
2  575045.12  1993          288.53
3  594056.47  1659          358.08
4  304896.48  2352          129.63
5  369782.05  2311          160.01


The `formula` lambda can reference any number of columns and return any expression that produces a Series or array of length `n`:

```python
# Discount percentage applied to price
Feature('final_price', dtype='computed',
        formula=lambda df: df['price'] * (1 - df['discount_rate']))

# Conditional logic
Feature('tier', dtype='computed',
        formula=lambda df: pd.cut(df['score'],
                                  bins=[0, 50, 80, 100],
                                  labels=['bronze', 'silver', 'gold']))
```

## Derived features

`derived=True` marks a feature whose value comes **entirely** from influences. It starts at zero and has no base distribution of its own.

Use `derived=True` when a column represents something that only exists because of a relationship — like revenue that is always `price × quantity` — and there is no sensible baseline to centre it on.

Derived features must be paired with at least one Influence targeting them. Additive per-unit effects (`"+N per unit"`) work naturally here because 0 × anything = 0, whereas percentage effects multiply the starting value (also 0), which produces nothing useful.

In [ ]:
bp = Blueprint(n=6, seed=42)
bp.add_feature(
    Feature('units_sold', dtype=int,   base=50, std=20, clip=(1, None)),
    Feature('revenue',    dtype=float, base=0,  std=0,  derived=True),
)
bp.add_influence(
    Influence('units_sold').on('revenue', effect='+99 per unit')
)

df = bp.emit()
print(df)
print()
print('revenue / units_sold:', (df['revenue'] / df['units_sold']).round(2).to_list())

   units_sold  revenue
0          56   5544.0
1          29   2871.0
2          65   6435.0
3          69   6831.0
4          11   1089.0
5          24   2376.0

revenue / units_sold: [99.0, 99.0, 99.0, 99.0, 99.0, 99.0]


## Nullable features

Any `Feature` can inject missing values by setting `nullable` to a fraction between 0 and 1. Blueprint randomly replaces that proportion of rows with `NaN` (numeric) or `None` (object/category columns).

In [ ]:
bp = Blueprint(n=10, seed=42)
bp.add_feature(
    Feature('phone', dtype='category',
            values=['+1-555-0100', '+1-555-0101', '+1-555-0102'],
            nullable=0.30),
    Feature('age', dtype=int, base=35, std=10, clip=(18, 80),
            nullable=0.20),
)

df = bp.emit()
print(df)
print()
print('phone NaN count:', df['phone'].isna().sum(), '/ 10')
print('age   NaN count:', df['age'].isna().sum(),   '/ 10')

         phone   age
0  +1-555-0100   NaN
1  +1-555-0102  39.0
2  +1-555-0101   NaN
3         None   NaN
4  +1-555-0101  35.0
5  +1-555-0102  33.0
6  +1-555-0100  28.0
7  +1-555-0102  47.0
8  +1-555-0100  33.0
9  +1-555-0100  31.0

phone NaN count: 1 / 10
age   NaN count: 3 / 10


When `nullable > 0`, a numeric column's dtype widens to `float64` because pandas integers can't hold `NaN`. Use `pd.array(..., dtype='Int64')` downstream if you need nullable integer columns.

## Feature modifiers

Modifiers are post-generation transforms that layer on top of the base distribution. They are applied **in the order they are added**, before the constructor `clip` is enforced.

| Method | What it does |
|---|---|
| `.trend(rate, style)` | Adds a row-indexed trend (linear or exponential growth/decay) |
| `.seasonality(period, amplitude, phase)` | Adds a sinusoidal seasonal pattern |
| `.noise(std, distribution)` | Adds random noise on top of the signal |
| `.round(decimals)` | Rounds to a fixed number of decimal places |
| `.clip(min, max)` | Applies a mid-pipeline clamp (separate from the constructor clip) |

Modifiers only apply to numeric features. They are silently skipped for categorical, boolean, and identity features.

### `.trend(rate, style)`

Adds a row-indexed trend to the generated values. Row `i` receives an additive adjustment proportional to `i`.

- `style="linear"` (default): adds `base × rate × i` to row `i`
- `style="exponential"`: multiplies the value by `(1 + rate)^i`
- `rate` can be negative for a downward trend

In [ ]:
# std=0 removes the base noise so only the trend is visible
bp = Blueprint(n=12, seed=42)
f = Feature('revenue', dtype=float, base=10_000, std=0)
f.trend(rate=0.02)  # 2% growth per row (linear)
bp.add_feature(f)

df = bp.emit()
print(df.round(2))

   revenue
0   10000.0
1   10200.0
2   10400.0
3   10600.0
4   10800.0
5   11000.0
6   11200.0
7   11400.0
8   11600.0
9   11800.0
10  12000.0
11  12200.0


### `.seasonality(period, amplitude, phase)`

Adds a sinusoidal wave to the values:

```
adjustment = amplitude × sin(2π × i / period + phase)
```

- `period` — number of rows per full cycle (e.g. 7 for weekly, 12 for monthly)
- `amplitude` — peak-to-peak height of the wave
- `phase` — phase shift in radians (shifts the wave left/right)

In [ ]:
# 12-row annual cycle with ±2000 amplitude
bp = Blueprint(n=12, seed=42)
f = Feature('visits', dtype=float, base=5_000, std=0)
f.seasonality(period=12, amplitude=2000, phase=0)
bp.add_feature(f)

df = bp.emit()
print(df.round(1))

    visits
0   5000.0
1   6000.0
2   6732.1
3   7000.0
4   6732.1
5   6000.0
6   5000.0
7   4000.0
8   3267.9
9   3000.0
10  3267.9
11  4000.0


### `.noise(std, distribution)`

Adds random noise on top of the current signal. Three distributions are available:

| `distribution` | Shape |
|---|---|
| `"gaussian"` (default) | Normal noise — concentrated near zero, tails extend further |
| `"uniform"` | Flat noise — each row gets a random offset in `[-std, +std]` |
| `"poisson"` | Discrete noise — useful for count data (integer offsets) |

In [ ]:
bp = Blueprint(n=8, seed=42)
f = Feature('sensor_reading', dtype=float, base=100, std=0)
f.noise(std=5, distribution='gaussian')
bp.add_feature(f)
df = bp.emit()
print('Gaussian noise (std=5):')
print(df.round(3))
print()

bp2 = Blueprint(n=8, seed=42)
f2 = Feature('jitter', dtype=float, base=100, std=0)
f2.noise(std=5, distribution='uniform')
bp2.add_feature(f2)
df2 = bp2.emit()
print('Uniform noise (std=5):')
print(df2.round(3))

Gaussian noise (std=5):
   sensor_reading
0         101.524
1          94.800
2         103.752
3         104.703
4          90.245
5          93.489
6         100.639
7          98.419

Uniform noise (std=5):
    jitter
0  102.740
1   99.389
2  103.586
3  101.974
4   95.942
5  104.756
6  102.611
7  102.861


### `.round(decimals)` and `.clip(min, max)`

`.round(decimals)` snaps values to a fixed number of decimal places. Use `decimals=0` for whole-number floats, or negative values to round to tens/hundreds.

`.clip(min, max)` as a **modifier** applies a mid-pipeline clamp, before the constructor `clip` fires. This lets you tighten bounds at a specific point in the modifier chain.

In [ ]:
# round() — useful for currency, percentages, or any fixed-precision output
bp = Blueprint(n=6, seed=42)
f = Feature('price', dtype=float, base=49.99, std=20)
f.round(decimals=2)
bp.add_feature(f)
df = bp.emit()
print('With round(2):')
print(df)
print()

# clip() as modifier — same semantics as constructor clip but chained
bp2 = Blueprint(n=6, seed=42)
f2 = Feature('score', dtype=float, base=100, std=30)
f2.clip(min=0, max=150)
bp2.add_feature(f2)
df2 = bp2.emit()
print('With clip(min=0, max=150):')
print(df2.round(2))

With round(2):
   price
0  56.08
1  29.19
2  65.00
3  68.80
4  10.97
5  23.95

With clip(min=0, max=150):
    score
0  109.14
1   68.80
2  122.51
3  128.22
4   41.47
5   60.93


## Modifier chaining and order of application

All modifier methods return `self`, so you can chain them in a single expression. The modifiers are applied in the order you add them — this matters because the output of each step becomes the input of the next.

**Typical order for time-series columns:** trend → seasonality → noise → round

1. **trend** — establishes the long-run direction
2. **seasonality** — overlays cyclical variation
3. **noise** — adds row-level randomness on top
4. **round** — snaps to the desired precision

The constructor `clip=(lo, hi)` fires last of all, after every modifier.

In [ ]:
bp = Blueprint(n=24, seed=42)

f = (
    Feature('monthly_sales', dtype=float, base=50_000, std=5_000)
    .trend(rate=0.005)                            # gentle upward drift
    .seasonality(period=12, amplitude=10_000)     # annual boom/bust
    .noise(std=1_500)                             # row-level randomness
    .round(decimals=2)                            # currency precision
)

bp.add_feature(f)

print('Modifiers applied:', [m['type'] for m in f.modifiers])
df = bp.emit()
print(df.head(12).to_string())

Modifiers applied: ['trend', 'seasonality', 'noise', 'round']
    monthly_sales
0        50881.09
1        49521.88
2        63710.97
3        66000.99
4        50524.18
5        50385.33
6        55351.67
7        44559.16
8        42487.38
9        36764.12
10       49160.70
11       53332.42


## Summary

| dtype | Key params | Output dtype |
|---|---|---|
| `float` | `base`, `std`, `clip` | `float64` |
| `int` | `base`, `std`, `clip` | `int64` |
| `"positive_float"` | `base`, `std` | `float64` |
| `"percentage"` | `base`, `std` | `float64` |
| `bool` | `p` | `bool` |
| `"category"` | `values`, `weights` | `CategoricalDtype` |
| `"id"` | `style`, `start`, `prefix`, `padding` | `object` / `int64` |
| `"datetime"` | `start`, `end` | `datetime64[ns]` |
| `"str"` | `template`, `pools` | `object` |
| `"computed"` | `formula` | depends on formula |
| `float` + `derived=True` | (from influences) | `float64` |

Any Feature can also take `nullable=<fraction>` to introduce missing values.

**What's next:**
- **Notebook 3** — Classes: define population segments and override feature parameters per segment
- **Notebook 4** — Influences: express causal relationships between columns
- **Notebook 5** — The dependency DAG: how Blueprint resolves evaluation order